# Lesson 03 — Inpainting: Removing Unwanted Objects

## Why This Lesson
Inpainting fills in masked regions by propagating information from surrounding pixels.
Watermark removal, object removal, scratch repair — all use inpainting.

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt

img = cv2.imread('sample.jpg')

# Create a mask — white = region to fill, black = keep
mask = np.zeros(img.shape[:2], dtype=np.uint8)

# Simulate a watermark/text overlay in the mask
cv2.putText(mask, 'WATERMARK', (50, img.shape[0]//2),
            cv2.FONT_HERSHEY_SIMPLEX, 2, 255, 8)

# Also add some scratch lines
cv2.line(mask, (0, 100), (img.shape[1], 120), 255, 4)
cv2.line(mask, (200, 0), (220, img.shape[0]), 255, 3)

# Apply the damage to the image
damaged = img.copy()
damaged[mask > 0] = [255, 255, 255]

# Inpaint: TELEA (fast marching) vs NS (Navier-Stokes fluid dynamics)
inpainted_telea = cv2.inpaint(damaged, mask, inpaintRadius=5, flags=cv2.INPAINT_TELEA)
inpainted_ns    = cv2.inpaint(damaged, mask, inpaintRadius=5, flags=cv2.INPAINT_NS)

fig, axes = plt.subplots(1, 4, figsize=(22, 6))
for ax, im, t in zip(axes,
    [img, damaged, inpainted_telea, inpainted_ns],
    ['Original', 'Damaged (watermark+scratches)', 'TELEA inpaint', 'NS inpaint']):
    ax.imshow(cv2.cvtColor(im, cv2.COLOR_BGR2RGB)); ax.set_title(t); ax.axis('off')
plt.suptitle('Inpainting fills masked regions from surrounding context', fontsize=12)
plt.show()

## Key Takeaway
INPAINT_TELEA is faster. INPAINT_NS gives better results for larger regions.
The quality depends entirely on how good your mask is — accurate mask = clean result.